USING PANDAS

In [ ]:
import pandas as pd
import numpy as np

# Create dataset
data = {
    'City': ['New York', 'London', 'Paris', 'Tokyo', 'Berlin', 'New York', 'London', 'Paris'],
    'Population': [8419000, 8982000, 2141000, 13960000, 3645000, 8419000, 8982000, 2141000],
    'Continent': ['North America', 'Europe', 'Europe', 'Asia', 'Europe', 'North America', 'Europe', 'Europe']
}

df = pd.DataFrame(data)

print("Original DataFrame:")
print(df)
print("\n")

# One-hot encode City
city_encoded = pd.get_dummies(df['City'], prefix='City')

# One-hot encode Continent
continent_encoded = pd.get_dummies(df['Continent'], prefix='Continent')

# Combine with original data
df_encoded = pd.concat([df, city_encoded, continent_encoded], axis=1)

print("After One-Hot Encoding:")
print(df_encoded)
print("\n")

# High cardinality example
df['UniqueID'] = np.arange(len(df))
high_cardinality_encoded = pd.get_dummies(df['UniqueID'], prefix='ID')

df_high_cardinality = pd.concat([df, high_cardinality_encoded], axis=1)

print("High Cardinality Example:")
print(df_high_cardinality.head())
print("\n")

# Handling missing values
df_missing = df.copy()
df_missing.loc[1, 'City'] = np.nan
df_missing.loc[3, 'Continent'] = np.nan

print("With Missing Values:")
print(df_missing)
print("\n")

# Fill missing values
df_missing['City'] = df_missing['City'].fillna('Unknown')
df_missing['Continent'] = df_missing['Continent'].fillna('Unknown')

# Encode again
df_missing_encoded = pd.get_dummies(
    df_missing,
    columns=['City', 'Continent'],
    prefix=['City', 'Continent']
)

print("After Handling Missing Values:")
print(df_missing_encoded)

Original DataFrame:
       City  Population      Continent
0  New York     8419000  North America
1    London     8982000         Europe
2     Paris     2141000         Europe
3     Tokyo    13960000           Asia
4    Berlin     3645000         Europe
5  New York     8419000  North America
6    London     8982000         Europe
7     Paris     2141000         Europe


After One-Hot Encoding:
       City  Population      Continent  City_Berlin  City_London  \
0  New York     8419000  North America        False        False   
1    London     8982000         Europe        False         True   
2     Paris     2141000         Europe        False        False   
3     Tokyo    13960000           Asia        False        False   
4    Berlin     3645000         Europe         True        False   
5  New York     8419000  North America        False        False   
6    London     8982000         Europe        False         True   
7     Paris     2141000         Europe        False        

USING SCIKIT-LEARN


In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Sample dataset
df = pd.DataFrame({
    'City': ['New York', 'London', 'Paris', 'Tokyo', np.nan],
    'Country': ['USA', 'UK', 'France', 'Japan', 'USA'],
    'Population': [8419000, 8982000, 2141000, 13960000, 5000000]
})

# Label Encoding for Country
label = LabelEncoder()
df['Country_encoded'] = label.fit_transform(df['Country'])

# Column Transformer
preprocessor = ColumnTransformer([
    ('onehot', OneHotEncoder(handle_unknown='ignore'), ['City'])
], remainder='passthrough')

# Pipeline
# Configure SimpleImputer to return a DataFrame
imputer_step = SimpleImputer(strategy='most_frequent').set_output(transform="pandas")
pipeline = Pipeline([
    ('imputer', imputer_step),
    ('preprocessor', preprocessor)
])

# Transform data
transformed_data = pipeline.fit_transform(df)

# Feature names
# Get all feature names from the ColumnTransformer, which correctly handles passthrough columns
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()

# Final DataFrame
df_encoded = pd.DataFrame(transformed_data, columns=feature_names)

print("Transformed DataFrame:")
print(df_encoded)
print("\n")

# Inverse transform example
# This assumes 'Country_encoded' is passed through and becomes a column in df_encoded
country_inverse = label.inverse_transform(df_encoded['remainder__Country_encoded'].astype(int))

print("Recovered Country Names:")
print(country_inverse)

Transformed DataFrame:
  onehot__City_London onehot__City_New York onehot__City_Paris  \
0                 0.0                   1.0                0.0   
1                 1.0                   0.0                0.0   
2                 0.0                   0.0                1.0   
3                 0.0                   0.0                0.0   
4                 1.0                   0.0                0.0   

  onehot__City_Tokyo remainder__Country remainder__Population  \
0                0.0                USA               8419000   
1                0.0                 UK               8982000   
2                0.0             France               2141000   
3                1.0              Japan              13960000   
4                0.0                USA               5000000   

  remainder__Country_encoded  
0                          3  
1                          2  
2                          0  
3                          1  
4                          3  


R